# Normal CH1/CH2 operation with anchor_free_reps + fixed external trigger

Purpose: inspect the REAL (non-diagnostic) pulse sequence on a scope --
normal CH1 laser/AOM drive and normal CH2 mw-on/mw-off gating -- but with
the `anchor_free_reps` fix applied, so multiple full off/on reference
cycles play per external trigger edge instead of needing one every single
cycle. Uses the real `rabi.setup_awg_sequences()`/`rabi.
_configure_external_trigger()` functions (not a reimplementation).

Background (see `notes.md`'s "spurious off-resonance/no-MW-near-sample
signal" entry): a spurious lock-in signal was traced to the `onceWaitTrig`
anchor's trigger-wait dead time, which delays exactly when each off/on
cycle resumes, once per cycle -- a real, reproducible glitch landing on
only one side of the reference. `anchor_free_reps` dilutes this by
listing that many off+on cycles in the sequence table before needing to
wrap back through the anchor (cheap -- each listing just references the
same already-uploaded arb data, not new waveform memory). CH1 and CH2
both wrap through their own anchors at the SAME `anchor_free_reps`
cadence, so they stay mutually aligned (an earlier attempt that applied
this to CH2 only, while CH1 used a different scheme, visibly misaligned
CH1 against the Sync/marker output on a scope -- fixed now).

This notebook has NO `ch1_hold_constant`/`ch2_hold_constant` -- both
channels run their real, normal content.

## Connect to the AWG + SDG1062X

In [25]:
import sys
sys.path.insert(0, "..")

import time
import numpy as np

import rabi
import ks33600a
import sdg1062x

awg = ks33600a.KS33600A(rabi.AWG_RESOURCE, debug=True)
sdg = sdg1062x.SDG1062X(rabi.SDG_RESOURCE, debug=True)
print("Connected to AWG and SDG1062X.")


Keysight 33600A: connected
*RST => +0,"No error"
*CLS => +0,"No error"
SOUR1:DATA:VOL:CLE => +0,"No error"
SOUR2:DATA:VOL:CLE => +0,"No error"
*RST
Siglent SDG1062X: connected
Connected to AWG and SDG1062X.


## Parameters

`MW_US` is a fixed representative pulse width (not swept -- this
notebook is a single-point scope check). `N_REPS`/`ANCHOR_FREE_REPS` are
kept small so individual cycles and anchor-wraps are easy to count on a
scope -- bump `N_REPS` to 250 and `ANCHOR_FREE_REPS` to 20 (the real
`run-ch2-constant` default) once the basic shape looks right.

In [46]:
LASER_US = 2.0
PRE_US = 1.0
MW_US = 2.0            # fixed representative tau_mw for this scope check
POST_US = 1.0
N_REPS = 5             # small -- real sweeps use 250 -- kept small so
                        # individual off/on cycles are easy to count

# 20000 caused a VisaIOError (VI_ERROR_TMO) writing CH1's DATA:SEQ --
# 40000 listed segments (2 per off+on pair) is a multi-megabyte SCPI
# command that the AWG couldn't process/hold. 50 worked. Trying 200 next
# -- close to (but comfortably under) the ~512-sequence-steps-per-channel
# spec estimated for this AWG series (512 - 1 anchor, /2 per pair =~ 255)
# -- to narrow down the real ceiling. If this fails too, back off further
# (e.g. 100); if it works, try pushing closer to 255.
ANCHOR_FREE_REPS = 250

CH1_VPP = 0.632
CH2_VPP = 5.0
CH2_OFFSET_V = 2.5

SEQUENCE_NAME_CH1 = "anchor_free_test_ch1"
SEQUENCE_NAME_CH2 = "anchor_free_test_ch2"


## Configure the external trigger at a literal, fixed 1 Hz

Directly sets the SDG1062X to a 1 Hz square wave -- one visible pulse
per second on the scope, no margin/ref_period_s calculation involved.

In [47]:
rep_us = LASER_US + PRE_US + MW_US + POST_US
ref_period_s = 2 * N_REPS * rep_us * ANCHOR_FREE_REPS * 1e-6

TRIGGER_HZ = 10.0  # literal, fixed -- one visible pulse per second

sdg.write("C1:BSWV WVTP,SQUARE")
sdg.write(f"C1:BSWV FRQ,{TRIGGER_HZ}")
sdg.write("C1:BSWV AMP,5")
sdg.write("C1:BSWV OFST,2.5")
sdg.write("C1:OUTP ON")

print(f"rep_us={rep_us:.3f} us")
print(f"one off+on cycle = {2 * N_REPS * rep_us * 1e-3:.3f} ms")
print(f"full anchor-to-anchor stretch ({ANCHOR_FREE_REPS} cycles) = {ref_period_s:.3f} s")
print(f"SDG1062X trigger fixed at {TRIGGER_HZ} Hz (one pulse per second)")


C1:BSWV WVTP,SQUARE
C1:BSWV FRQ,10.0
C1:BSWV AMP,5
C1:BSWV OFST,2.5
C1:OUTP ON
rep_us=6.000 us
one off+on cycle = 0.060 ms
full anchor-to-anchor stretch (250 cycles) = 0.015 s
SDG1062X trigger fixed at 10.0 Hz (one pulse per second)


## Build the sequence: normal CH1/CH2, with anchor_free_reps

Uses the real `rabi.setup_awg_sequences()` -- `ch1_hold_constant` and
`ch2_hold_constant` are both left at their default `False`, so this is
exactly what a real `cmd_run()` sweep point would produce, just with
`anchor_free_reps` applied.

In [48]:
rabi.setup_awg_sequences(
    awg, MW_US, N_REPS, LASER_US, PRE_US, POST_US,
    sequence_name_ch1=SEQUENCE_NAME_CH1,
    sequence_name_ch2=SEQUENCE_NAME_CH2,
    ch1_vpp=CH1_VPP, ch2_vpp=CH2_VPP, ch2_offset_v=CH2_OFFSET_V,
    anchor_free_reps=ANCHOR_FREE_REPS,
)
print("Sequence uploaded and running (normal CH1/CH2 content).")


SOUR1:DATA:VOL:CLE => +0,"No error"
SOUR2:DATA:VOL:CLE => +0,"No error"
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 1
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 1
DATA:SEQ #513558"anchor_free_test_ch1","anchor",1,onceWaitTrig,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"rep",5,repeat,maintain,10,"

## Diagnostic: confirm the instrument's actual state

In [ ]:
print("OUTP1?", awg.query("OUTP1?"))
print("OUTP2?", awg.query("OUTP2?"))
print("SOUR1:FUNC?", awg.query("SOUR1:FUNC?"))
print("SOUR2:FUNC?", awg.query("SOUR2:FUNC?"))
print("SOUR1:FUNC:ARB?", awg.query("SOUR1:FUNC:ARB?"))
print("SOUR2:FUNC:ARB?", awg.query("SOUR2:FUNC:ARB?"))
print("SOUR1:VOLT?", awg.query("SOUR1:VOLT?"))
print("SOUR2:VOLT?", awg.query("SOUR2:VOLT?"))
print("SOUR2:VOLT:OFFS?", awg.query("SOUR2:VOLT:OFFS?"))
print("TRIG1:SOUR?", awg.query("TRIG1:SOUR?"))
print("TRIG2:SOUR?", awg.query("TRIG2:SOUR?"))
print("OUTPut:SYNC:SOURce?", awg.query("OUTPut:SYNC:SOURce?"))
print("SYST:ERR?", awg.query("SYST:ERR?"))


## What to check on the oscilloscope

- **First, capture right around a trigger edge** (use the SDG1062X output
  or the AWG's Ext Trig input as the scope's trigger source) to confirm
  both channels release from their anchors together and start the first
  off+on cycle aligned, same as before.
- **Then move the scope's timebase/capture window to somewhere in the
  MIDDLE of the long `ANCHOR_FREE_REPS` stretch** (seconds after the
  trigger edge, well before the next one is due) and check CH1 vs. Sync
  alignment there too. This is the real test: with no retriggering
  happening anywhere nearby, has CH1 drifted relative to CH2/Sync by
  this point, or does it stay locked throughout the entire multi-second
  free-run?
- **Sweep the capture window across the full stretch** (early / middle /
  late, all before the next trigger edge) to see whether any drift is
  gradual (a slow phase creep, consistent with a small clock-rate
  mismatch between channels) or if alignment simply holds throughout with
  no retriggering needed at all.
- **CH2 and Sync** should look exactly like before (this was already
  confirmed clean) -- the point of this run is specifically to observe
  CH1's behavior over a much longer, mostly trigger-free stretch.

## Stop / disconnect

In [ ]:
awg.write("OUTPUT1 OFF")
awg.write("OUTPUT2 OFF")
awg.close()
sdg.write("C1:OUTP OFF")
sdg.close()
print("AWG and SDG1062X outputs off, connections closed.")
